# Dropout Prediction — Improved Models (v2)

Improvements over v1:
- **Pipeline objects** throughout (imputer → scaler → model in a single `Pipeline`)
- **Richer feature engineering**: session statistics (mean, std), engagement score, grade acceleration, at-risk flag, interaction terms
- **Different models per stage**: GradientBoosting (S1), ExtraTrees (S2), tuned HGB (S3), soft VotingClassifier (S4)
- **Threshold optimisation**: CV-optimal threshold per stage (maximise macro-F1)
- **Full evaluation**: classification report + ROC-AUC for every stage

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier,
    VotingClassifier
)
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, f1_score
)
from sklearn.model_selection import cross_val_predict, StratifiedKFold

GRADE_MAP = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
YEAR_MAP  = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}


def _base_encode(df):
    """Shared ordinal encoding applied to every stage."""
    df = df.copy()
    df['external_flag'] = df['External'].map({'Y': 1, 'N': 0})
    df['year_num']      = df['Year'].str.lower().map(YEAR_MAP)
    return df


def _grade_cols(df, cols):
    """Map letter-grade columns to numeric in-place."""
    for col in cols:
        key = col.replace(' ', '_') + '_num'
        df[key] = df[col].str.upper().map(GRADE_MAP)
    return df


def _optimal_threshold(y_true, probas):
    """Grid-search threshold that maximises macro-F1."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(0.1, 0.9, 81):
        preds = (probas >= t).astype(int)
        f = f1_score(y_true, preds, average='macro', zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t


print('Shared utilities loaded.')

Shared utilities loaded.


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Stage 1  — GradientBoostingClassifier  (after session 2 / test 1)
# New vs v1: richer features + Pipeline + GBC replaces plain LogReg
# ─────────────────────────────────────────────────────────────────────────────
def predict_dropout1():
    train = _grade_cols(_base_encode(pd.read_csv('ND26_dropout.csv')), ['test 1'])

    # Feature engineering
    train['session_mean']    = train[['session 1', 'session 2']].mean(axis=1)
    train['session_total']   = train['session 1'].fillna(0) + train['session 2'].fillna(0)
    train['engagement']      = train['forum Q'].fillna(0) + train['forum A'].fillna(0)
    train['yr_grade_ix']     = train['year_num'] * train['test_1_num']   # interaction
    train['grade_eng_ix']    = train['test_1_num'] * train['engagement'] # interaction
    train['y']               = train['dropout'].map({'Y': 1, 'N': 0})

    feats = [
        'external_flag', 'year_num',
        'session 1', 'session 2', 'session_mean', 'session_total',
        'test_1_num', 'forum Q', 'forum A', 'engagement',
        'office hour visits', 'yr_grade_ix', 'grade_eng_ix',
    ]

    pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, min_samples_leaf=20, random_state=42
        )),
    ])
    pipe.fit(train[feats], train['y'])

    test = _grade_cols(_base_encode(pd.read_csv('./data/entry_dropout1.csv')), ['test 1'])
    test['session_mean']    = test[['session 1', 'session 2']].mean(axis=1)
    test['session_total']   = test['session 1'].fillna(0) + test['session 2'].fillna(0)
    test['engagement']      = test['forum Q'].fillna(0) + test['forum A'].fillna(0)
    test['yr_grade_ix']     = test['year_num'] * test['test_1_num']
    test['grade_eng_ix']    = test['test_1_num'] * test['engagement']

    probas = pipe.predict_proba(test[feats])[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Stage 2  — ExtraTreesClassifier  (after session 4 / test 2)
# New vs v1: session std + grade trend + engagement rate + Pipeline
# ─────────────────────────────────────────────────────────────────────────────
def predict_dropout2():
    sess_cols2 = ['session 1', 'session 2', 'session 3', 'session 4']

    train = _grade_cols(
        _base_encode(pd.read_csv('ND26_dropout.csv')),
        ['test 1', 'test 2']
    )
    train['session_total']  = train[sess_cols2].fillna(0).sum(axis=1)
    train['session_mean']   = train[sess_cols2].mean(axis=1)
    train['session_std']    = train[sess_cols2].std(axis=1)          # NEW: consistency
    train['engagement']     = train['forum Q'].fillna(0) + train['forum A'].fillna(0)
    train['eng_per_session']= train['engagement'] / (train['session_total'] + 1)
    train['grade_mean']     = train[['test_1_num', 'test_2_num']].mean(axis=1)
    train['grade_delta']    = train['test_2_num'] - train['test_1_num']  # trend
    train['at_risk']        = (train['grade_mean'] < 2).astype(int)      # NEW: flag
    train['yr_grade_ix']    = train['year_num'] * train['grade_mean']
    train['y']              = train['dropout'].map({'Y': 1, 'N': 0})
    train = train[train['session 3'].notna() | train['test 2'].notna()]

    feats = [
        'external_flag', 'year_num',
        'session 1', 'session 2', 'session 3', 'session 4',
        'session_mean', 'session_std', 'session_total',
        'test_1_num', 'test_2_num', 'grade_mean', 'grade_delta', 'at_risk',
        'forum Q', 'forum A', 'engagement', 'eng_per_session',
        'office hour visits', 'yr_grade_ix',
    ]

    # ExtraTrees: no scaling needed, natively handles missing after impute
    pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('clf', ExtraTreesClassifier(
            n_estimators=400, max_depth=10, min_samples_leaf=5,
            class_weight='balanced_subsample', random_state=42, n_jobs=-1
        )),
    ])
    pipe.fit(train[feats], train['y'])

    test = _grade_cols(
        _base_encode(pd.read_csv('./data/entry_dropout2.csv')),
        ['test 1', 'test 2']
    )
    test['session_total']   = test[sess_cols2].fillna(0).sum(axis=1)
    test['session_mean']    = test[sess_cols2].mean(axis=1)
    test['session_std']     = test[sess_cols2].std(axis=1)
    test['engagement']      = test['forum Q'].fillna(0) + test['forum A'].fillna(0)
    test['eng_per_session'] = test['engagement'] / (test['session_total'] + 1)
    test['grade_mean']      = test[['test_1_num', 'test_2_num']].mean(axis=1)
    test['grade_delta']     = test['test_2_num'] - test['test_1_num']
    test['at_risk']         = (test['grade_mean'] < 2).astype(int)
    test['yr_grade_ix']     = test['year_num'] * test['grade_mean']

    probas = pipe.predict_proba(test[feats])[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Stage 3  — Tuned HistGradientBoosting  (after session 5 / test 3)
# New vs v1: grade acceleration + session trend + per-test z-score features
# ─────────────────────────────────────────────────────────────────────────────
def predict_dropout3():
    sess_cols3 = ['session 1', 'session 2', 'session 3', 'session 4', 'session 5']
    grade_cols3 = ['test 1', 'test 2', 'test 3']

    train = _grade_cols(
        _base_encode(pd.read_csv('ND26_dropout.csv')),
        grade_cols3
    )
    train['session_total']   = train[sess_cols3].fillna(0).sum(axis=1)
    train['session_mean']    = train[sess_cols3].mean(axis=1)
    train['session_std']     = train[sess_cols3].std(axis=1)
    train['session_trend']   = train['session 5'].fillna(0) - train['session 1'].fillna(0)
    train['engagement']      = train['forum Q'].fillna(0) + train['forum A'].fillna(0)
    train['eng_per_session'] = train['engagement'] / (train['session_total'] + 1)
    train['grade_mean']      = train[['test_1_num','test_2_num','test_3_num']].mean(axis=1)
    train['grade_delta']     = train['test_3_num'] - train['test_1_num']   # overall trend
    train['grade_accel']     = (                                            # NEW: acceleration
        (train['test_3_num'] - train['test_2_num']) -
        (train['test_2_num'] - train['test_1_num'])
    )
    train['at_risk']         = (train['grade_mean'] < 2).astype(int)
    train['high_performer']  = (train['grade_mean'] >= 4).astype(int)      # NEW
    train['yr_grade_ix']     = train['year_num'] * train['grade_mean']
    train['y']               = train['dropout'].map({'Y': 1, 'N': 0})
    train = train[train['session 5'].notna() | train['test 3'].notna()]

    feats = [
        'external_flag', 'year_num',
        'session 1', 'session 2', 'session 3', 'session 4', 'session 5',
        'session_mean', 'session_std', 'session_total', 'session_trend',
        'test_1_num', 'test_2_num', 'test_3_num',
        'grade_mean', 'grade_delta', 'grade_accel', 'at_risk', 'high_performer',
        'forum Q', 'forum A', 'engagement', 'eng_per_session',
        'office hour visits', 'yr_grade_ix',
    ]

    # HGB natively handles NaN — only imputer needed for robustness on edge cases
    pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('clf', HistGradientBoostingClassifier(
            max_iter=300, max_depth=6, learning_rate=0.05,
            min_samples_leaf=20, l2_regularization=0.1,
            class_weight='balanced', random_state=42
        )),
    ])
    pipe.fit(train[feats], train['y'])

    test = _grade_cols(
        _base_encode(pd.read_csv('./data/entry_dropout3.csv')),
        grade_cols3
    )
    test['session_total']   = test[sess_cols3].fillna(0).sum(axis=1)
    test['session_mean']    = test[sess_cols3].mean(axis=1)
    test['session_std']     = test[sess_cols3].std(axis=1)
    test['session_trend']   = test['session 5'].fillna(0) - test['session 1'].fillna(0)
    test['engagement']      = test['forum Q'].fillna(0) + test['forum A'].fillna(0)
    test['eng_per_session'] = test['engagement'] / (test['session_total'] + 1)
    test['grade_mean']      = test[['test_1_num','test_2_num','test_3_num']].mean(axis=1)
    test['grade_delta']     = test['test_3_num'] - test['test_1_num']
    test['grade_accel']     = (
        (test['test_3_num'] - test['test_2_num']) -
        (test['test_2_num'] - test['test_1_num'])
    )
    test['at_risk']         = (test['grade_mean'] < 2).astype(int)
    test['high_performer']  = (test['grade_mean'] >= 4).astype(int)
    test['yr_grade_ix']     = test['year_num'] * test['grade_mean']

    probas = pipe.predict_proba(test[feats])[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Stage 4  — Soft VotingClassifier  (end-of-year, all features)
# New vs v1: VotingClassifier (soft) replaces StackingClassifier,
#            coursework-vs-test gap features added
# ─────────────────────────────────────────────────────────────────────────────
def predict_dropout4():
    sess_cols4   = ['session 1','session 2','session 3','session 4','session 5','session 6']
    test_cols    = ['test 1', 'test 2', 'test 3']
    cw_cols      = ['ind cw', 'group cw']
    all_grade_c  = test_cols + cw_cols + ['final grade']

    train = _grade_cols(_base_encode(pd.read_csv('ND26_dropout.csv')), all_grade_c)
    num_tests = ['test_1_num','test_2_num','test_3_num']
    num_cw    = ['ind_cw_num','group_cw_num']

    train['session_total']   = train[sess_cols4].fillna(0).sum(axis=1)
    train['session_mean']    = train[sess_cols4].mean(axis=1)
    train['session_std']     = train[sess_cols4].std(axis=1)
    train['session_trend']   = train['session 6'].fillna(0) - train['session 1'].fillna(0)
    train['engagement']      = train['forum Q'].fillna(0) + train['forum A'].fillna(0)
    train['eng_per_session'] = train['engagement'] / (train['session_total'] + 1)
    train['test_mean']       = train[num_tests].mean(axis=1)
    train['cw_mean']         = train[num_cw].mean(axis=1)
    train['grade_mean']      = train[num_tests + num_cw + ['final_grade_num']].mean(axis=1)
    train['grade_delta']     = train['final_grade_num'] - train['test_1_num']
    train['grade_accel']     = (
        (train['test_3_num'] - train['test_2_num']) -
        (train['test_2_num'] - train['test_1_num'])
    )
    train['cw_vs_test']      = train['cw_mean'] - train['test_mean']  # NEW: coursework gap
    train['final_vs_init']   = train['final_grade_num'] - train['test_1_num']
    train['at_risk']         = (train['grade_mean'] < 2).astype(int)
    train['high_performer']  = (train['grade_mean'] >= 4).astype(int)
    train['yr_grade_ix']     = train['year_num'] * train['grade_mean']
    train['y']               = train['dropout'].map({'Y': 1, 'N': 0})
    train = train[train['session 6'].notna() | train['ind cw'].notna()]

    feats = [
        'external_flag', 'year_num',
        'session 1','session 2','session 3','session 4','session 5','session 6',
        'session_mean','session_std','session_total','session_trend',
        'test_1_num','test_2_num','test_3_num','ind_cw_num','group_cw_num','final_grade_num',
        'test_mean','cw_mean','grade_mean','grade_delta','grade_accel',
        'cw_vs_test','final_vs_init','at_risk','high_performer',
        'forum Q','forum A','engagement','eng_per_session',
        'office hour visits','yr_grade_ix',
    ]

    # Soft VotingClassifier: averages predicted probabilities
    pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('clf', VotingClassifier(
            estimators=[
                ('lr',  LogisticRegression(max_iter=1000, C=0.3,
                                           class_weight='balanced', random_state=42)),
                ('et',  ExtraTreesClassifier(n_estimators=300, max_depth=8,
                                             class_weight='balanced_subsample',
                                             random_state=42, n_jobs=-1)),
                ('hgb', HistGradientBoostingClassifier(max_iter=200, max_depth=5,
                                                        learning_rate=0.05,
                                                        class_weight='balanced',
                                                        random_state=42)),
            ],
            voting='soft',
        )),
    ])
    pipe.fit(train[feats], train['y'])

    test = _grade_cols(_base_encode(pd.read_csv('./data/entry_dropout4.csv')), all_grade_c)
    test['session_total']   = test[sess_cols4].fillna(0).sum(axis=1)
    test['session_mean']    = test[sess_cols4].mean(axis=1)
    test['session_std']     = test[sess_cols4].std(axis=1)
    test['session_trend']   = test['session 6'].fillna(0) - test['session 1'].fillna(0)
    test['engagement']      = test['forum Q'].fillna(0) + test['forum A'].fillna(0)
    test['eng_per_session'] = test['engagement'] / (test['session_total'] + 1)
    test['test_mean']       = test[num_tests].mean(axis=1)
    test['cw_mean']         = test[num_cw].mean(axis=1)
    test['grade_mean']      = test[num_tests + num_cw + ['final_grade_num']].mean(axis=1)
    test['grade_delta']     = test['final_grade_num'] - test['test_1_num']
    test['grade_accel']     = (
        (test['test_3_num'] - test['test_2_num']) -
        (test['test_2_num'] - test['test_1_num'])
    )
    test['cw_vs_test']      = test['cw_mean'] - test['test_mean']
    test['final_vs_init']   = test['final_grade_num'] - test['test_1_num']
    test['at_risk']         = (test['grade_mean'] < 2).astype(int)
    test['high_performer']  = (test['grade_mean'] >= 4).astype(int)
    test['yr_grade_ix']     = test['year_num'] * test['grade_mean']

    probas = pipe.predict_proba(test[feats])[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [6]:
print('Stage 1:', predict_dropout1())
print('Stage 2:', predict_dropout2())
print('Stage 3:', predict_dropout3())
print('Stage 4:', predict_dropout4())

Stage 1: ['Y', 'N', 'N', 'Y', 'N', 'Y', 'Y', 'N', 'N', 'Y']
Stage 2: ['Y', 'Y', 'N', 'N', 'N', 'N', 'N', 'Y', 'Y', 'Y']
Stage 3: ['Y', 'Y', 'Y', 'N', 'N', 'Y', 'N', 'Y', 'N', 'N']
Stage 4: ['N', 'Y', 'Y', 'N', 'Y', 'N', 'Y', 'Y', 'N', 'N']


In [7]:
# ═════════════════════════════════════════════════════════════════════════════
# Cross-Validation Evaluation  —  all four stages
# ═════════════════════════════════════════════════════════════════════════════
train_full = pd.read_csv('ND26_dropout.csv')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Stage 1 ──────────────────────────────────────────────────────────────────
t1 = _grade_cols(_base_encode(train_full.copy()), ['test 1'])
t1['session_mean']    = t1[['session 1', 'session 2']].mean(axis=1)
t1['session_total']   = t1['session 1'].fillna(0) + t1['session 2'].fillna(0)
t1['engagement']      = t1['forum Q'].fillna(0) + t1['forum A'].fillna(0)
t1['yr_grade_ix']     = t1['year_num'] * t1['test_1_num']
t1['grade_eng_ix']    = t1['test_1_num'] * t1['engagement']
t1['y']               = t1['dropout'].map({'Y': 1, 'N': 0})

feats1 = [
    'external_flag', 'year_num',
    'session 1', 'session 2', 'session_mean', 'session_total',
    'test_1_num', 'forum Q', 'forum A', 'engagement',
    'office hour visits', 'yr_grade_ix', 'grade_eng_ix',
]
pipe1 = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler()),
    ('clf', GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, min_samples_leaf=20, random_state=42
    )),
])
X1, y1 = t1[feats1], t1['y'].values
p1_cv  = cross_val_predict(pipe1, X1, y1, cv=cv, method='predict_proba')[:, 1]
t1_opt = _optimal_threshold(y1, p1_cv)
y1_pred = (p1_cv >= t1_opt).astype(int)

print('=' * 62)
print(f'STAGE 1 — GradientBoosting   (after session 2 / test 1)')
print(f'          Optimal threshold: {t1_opt:.2f}')
print('=' * 62)
print(f'Accuracy : {accuracy_score(y1, y1_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y1, p1_cv):.4f}\n')
print(classification_report(y1, y1_pred, target_names=['No Dropout (N)', 'Dropout (Y)']))

# ── Stage 2 ──────────────────────────────────────────────────────────────────
sess_cols2 = ['session 1', 'session 2', 'session 3', 'session 4']
t2 = _grade_cols(_base_encode(train_full.copy()), ['test 1', 'test 2'])
t2['session_total']   = t2[sess_cols2].fillna(0).sum(axis=1)
t2['session_mean']    = t2[sess_cols2].mean(axis=1)
t2['session_std']     = t2[sess_cols2].std(axis=1)
t2['engagement']      = t2['forum Q'].fillna(0) + t2['forum A'].fillna(0)
t2['eng_per_session'] = t2['engagement'] / (t2['session_total'] + 1)
t2['grade_mean']      = t2[['test_1_num', 'test_2_num']].mean(axis=1)
t2['grade_delta']     = t2['test_2_num'] - t2['test_1_num']
t2['at_risk']         = (t2['grade_mean'] < 2).astype(int)
t2['yr_grade_ix']     = t2['year_num'] * t2['grade_mean']
t2['y']               = t2['dropout'].map({'Y': 1, 'N': 0})
t2 = t2[t2['session 3'].notna() | t2['test 2'].notna()]

feats2 = [
    'external_flag', 'year_num',
    'session 1', 'session 2', 'session 3', 'session 4',
    'session_mean', 'session_std', 'session_total',
    'test_1_num', 'test_2_num', 'grade_mean', 'grade_delta', 'at_risk',
    'forum Q', 'forum A', 'engagement', 'eng_per_session',
    'office hour visits', 'yr_grade_ix',
]
pipe2 = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('clf', ExtraTreesClassifier(
        n_estimators=400, max_depth=10, min_samples_leaf=5,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    )),
])
X2, y2 = t2[feats2], t2['y'].values
p2_cv  = cross_val_predict(pipe2, X2, y2, cv=cv, method='predict_proba')[:, 1]
t2_opt = _optimal_threshold(y2, p2_cv)
y2_pred = (p2_cv >= t2_opt).astype(int)

print('=' * 62)
print(f'STAGE 2 — ExtraTrees         (after session 4 / test 2)')
print(f'          Optimal threshold: {t2_opt:.2f}')
print('=' * 62)
print(f'Accuracy : {accuracy_score(y2, y2_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y2, p2_cv):.4f}\n')
print(classification_report(y2, y2_pred, target_names=['No Dropout (N)', 'Dropout (Y)']))

# ── Stage 3 ──────────────────────────────────────────────────────────────────
sess_cols3 = ['session 1','session 2','session 3','session 4','session 5']
t3 = _grade_cols(_base_encode(train_full.copy()), ['test 1', 'test 2', 'test 3'])
t3['session_total']   = t3[sess_cols3].fillna(0).sum(axis=1)
t3['session_mean']    = t3[sess_cols3].mean(axis=1)
t3['session_std']     = t3[sess_cols3].std(axis=1)
t3['session_trend']   = t3['session 5'].fillna(0) - t3['session 1'].fillna(0)
t3['engagement']      = t3['forum Q'].fillna(0) + t3['forum A'].fillna(0)
t3['eng_per_session'] = t3['engagement'] / (t3['session_total'] + 1)
t3['grade_mean']      = t3[['test_1_num','test_2_num','test_3_num']].mean(axis=1)
t3['grade_delta']     = t3['test_3_num'] - t3['test_1_num']
t3['grade_accel']     = (
    (t3['test_3_num'] - t3['test_2_num']) -
    (t3['test_2_num'] - t3['test_1_num'])
)
t3['at_risk']         = (t3['grade_mean'] < 2).astype(int)
t3['high_performer']  = (t3['grade_mean'] >= 4).astype(int)
t3['yr_grade_ix']     = t3['year_num'] * t3['grade_mean']
t3['y']               = t3['dropout'].map({'Y': 1, 'N': 0})
t3 = t3[t3['session 5'].notna() | t3['test 3'].notna()]

feats3 = [
    'external_flag', 'year_num',
    'session 1','session 2','session 3','session 4','session 5',
    'session_mean','session_std','session_total','session_trend',
    'test_1_num','test_2_num','test_3_num',
    'grade_mean','grade_delta','grade_accel','at_risk','high_performer',
    'forum Q','forum A','engagement','eng_per_session',
    'office hour visits','yr_grade_ix',
]
pipe3 = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('clf', HistGradientBoostingClassifier(
        max_iter=300, max_depth=6, learning_rate=0.05,
        min_samples_leaf=20, l2_regularization=0.1,
        class_weight='balanced', random_state=42
    )),
])
X3, y3 = t3[feats3], t3['y'].values
p3_cv  = cross_val_predict(pipe3, X3, y3, cv=cv, method='predict_proba')[:, 1]
t3_opt = _optimal_threshold(y3, p3_cv)
y3_pred = (p3_cv >= t3_opt).astype(int)

print('=' * 62)
print(f'STAGE 3 — HistGradientBoosting (after session 5 / test 3)')
print(f'          Optimal threshold: {t3_opt:.2f}')
print('=' * 62)
print(f'Accuracy : {accuracy_score(y3, y3_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y3, p3_cv):.4f}\n')
print(classification_report(y3, y3_pred, target_names=['No Dropout (N)', 'Dropout (Y)']))

# ── Stage 4 ──────────────────────────────────────────────────────────────────
sess_cols4  = ['session 1','session 2','session 3','session 4','session 5','session 6']
all_grade_c = ['test 1','test 2','test 3','ind cw','group cw','final grade']
num_tests   = ['test_1_num','test_2_num','test_3_num']
num_cw      = ['ind_cw_num','group_cw_num']

t4 = _grade_cols(_base_encode(train_full.copy()), all_grade_c)
t4['session_total']   = t4[sess_cols4].fillna(0).sum(axis=1)
t4['session_mean']    = t4[sess_cols4].mean(axis=1)
t4['session_std']     = t4[sess_cols4].std(axis=1)
t4['session_trend']   = t4['session 6'].fillna(0) - t4['session 1'].fillna(0)
t4['engagement']      = t4['forum Q'].fillna(0) + t4['forum A'].fillna(0)
t4['eng_per_session'] = t4['engagement'] / (t4['session_total'] + 1)
t4['test_mean']       = t4[num_tests].mean(axis=1)
t4['cw_mean']         = t4[num_cw].mean(axis=1)
t4['grade_mean']      = t4[num_tests + num_cw + ['final_grade_num']].mean(axis=1)
t4['grade_delta']     = t4['final_grade_num'] - t4['test_1_num']
t4['grade_accel']     = (
    (t4['test_3_num'] - t4['test_2_num']) -
    (t4['test_2_num'] - t4['test_1_num'])
)
t4['cw_vs_test']      = t4['cw_mean'] - t4['test_mean']
t4['final_vs_init']   = t4['final_grade_num'] - t4['test_1_num']
t4['at_risk']         = (t4['grade_mean'] < 2).astype(int)
t4['high_performer']  = (t4['grade_mean'] >= 4).astype(int)
t4['yr_grade_ix']     = t4['year_num'] * t4['grade_mean']
t4['y']               = t4['dropout'].map({'Y': 1, 'N': 0})
t4 = t4[t4['session 6'].notna() | t4['ind cw'].notna()]

feats4 = [
    'external_flag', 'year_num',
    'session 1','session 2','session 3','session 4','session 5','session 6',
    'session_mean','session_std','session_total','session_trend',
    'test_1_num','test_2_num','test_3_num','ind_cw_num','group_cw_num','final_grade_num',
    'test_mean','cw_mean','grade_mean','grade_delta','grade_accel',
    'cw_vs_test','final_vs_init','at_risk','high_performer',
    'forum Q','forum A','engagement','eng_per_session',
    'office hour visits','yr_grade_ix',
]
pipe4 = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler()),
    ('clf', VotingClassifier(
        estimators=[
            ('lr',  LogisticRegression(max_iter=1000, C=0.3,
                                       class_weight='balanced', random_state=42)),
            ('et',  ExtraTreesClassifier(n_estimators=300, max_depth=8,
                                         class_weight='balanced_subsample',
                                         random_state=42, n_jobs=-1)),
            ('hgb', HistGradientBoostingClassifier(max_iter=200, max_depth=5,
                                                    learning_rate=0.05,
                                                    class_weight='balanced',
                                                    random_state=42)),
        ],
        voting='soft',
    )),
])
X4, y4 = t4[feats4], t4['y'].values
p4_cv  = cross_val_predict(pipe4, X4, y4, cv=cv, method='predict_proba')[:, 1]
t4_opt = _optimal_threshold(y4, p4_cv)
y4_pred = (p4_cv >= t4_opt).astype(int)

print('=' * 62)
print(f'STAGE 4 — Soft VotingClassifier (end-of-year, all features)')
print(f'          Optimal threshold: {t4_opt:.2f}')
print('=' * 62)
print(f'Accuracy : {accuracy_score(y4, y4_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y4, p4_cv):.4f}\n')
print(classification_report(y4, y4_pred, target_names=['No Dropout (N)', 'Dropout (Y)']))

STAGE 1 — GradientBoosting   (after session 2 / test 1)
          Optimal threshold: 0.52
Accuracy : 0.9521
ROC-AUC  : 0.9911

                precision    recall  f1-score   support

No Dropout (N)       0.95      0.97      0.96     13805
   Dropout (Y)       0.96      0.94      0.95     11944

      accuracy                           0.95     25749
     macro avg       0.95      0.95      0.95     25749
  weighted avg       0.95      0.95      0.95     25749

STAGE 2 — ExtraTrees         (after session 4 / test 2)
          Optimal threshold: 0.53
Accuracy : 0.9479
ROC-AUC  : 0.9881

                precision    recall  f1-score   support

No Dropout (N)       0.96      0.96      0.96     13805
   Dropout (Y)       0.93      0.94      0.93      9004

      accuracy                           0.95     22809
     macro avg       0.95      0.95      0.95     22809
  weighted avg       0.95      0.95      0.95     22809

STAGE 3 — HistGradientBoosting (after session 5 / test 3)
          

In [8]:
# ── Summary comparison table ──────────────────────────────────────────────────
from sklearn.metrics import f1_score as _f1

rows = []
for stage, y_true, y_pred, probas in [
    (1, y1, y1_pred, p1_cv),
    (2, y2, y2_pred, p2_cv),
    (3, y3, y3_pred, p3_cv),
    (4, y4, y4_pred, p4_cv),
]:
    rows.append({
        'Stage'   : stage,
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'ROC-AUC' : round(roc_auc_score(y_true, probas), 4),
        'Macro-F1': round(_f1(y_true, y_pred, average='macro'), 4),
        'Dropout Recall': round(_f1(y_true, y_pred, average=None)[1], 4),
    })

summary = pd.DataFrame(rows).set_index('Stage')
print('\n── v2 Model Summary ──')
print(summary.to_string())


── v2 Model Summary ──
       Accuracy  ROC-AUC  Macro-F1  Dropout Recall
Stage                                             
1        0.9521   0.9911    0.9518          0.9478
2        0.9479   0.9881    0.9455          0.9341
3        0.9703   0.9949    0.9609          0.9417
4        0.9824   0.9962    0.9569          0.9238
